In [1]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
import torch.optim as optim
import torch.distributions as distributions 

import numpy  as np
import matplotlib.pyplot as plt 
import gymnasium as gym 

In [2]:
test_env = gym.make("CartPole-v1")
train_env = gym.make("CartPole-v1")

In [3]:
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout = 0.0):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim)
        )
        
    def forward(self, x):
        x = self.net(x)
        return x

In [4]:
class ActorCritic(nn.Module):
    def __init__(self, actor, critic):
        super().__init__()
        self.actor = actor 
        self.critic = critic 
    
    def forward(self, x):
        action_pred = self.actor(x)
        value_pred = self.critic(x)
        return action_pred, value_pred 


In [5]:
def init_weight(m):
    if type(m) == nn.Linear:
        torch.nn.init.xavier_normal_(m.weight)
        m.bias.data.fill_(0)


In [6]:
INPUT_DIM = train_env.observation_space.shape[0]
HIDDEN_DIM = 64
OUTPUT_DIM = test_env.action_space.n 

actor = MLP(INPUT_DIM, HIDDEN_DIM, OUTPUT_DIM)
critic = MLP(INPUT_DIM, HIDDEN_DIM, 1)

policy = ActorCritic(actor, critic)
policy.apply(init_weight)
policy

ActorCritic(
  (actor): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.0, inplace=False)
      (3): Linear(in_features=64, out_features=2, bias=True)
    )
  )
  (critic): MLP(
    (net): Sequential(
      (0): Linear(in_features=4, out_features=64, bias=True)
      (1): ReLU()
      (2): Dropout(p=0.0, inplace=False)
      (3): Linear(in_features=64, out_features=1, bias=True)
    )
  )
)

In [7]:
optimizer = optim.Adam(policy.parameters(), lr = 1.5e-3)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.0015
    maximize: False
    weight_decay: 0
)

In [8]:
def train(env, policy, optimizer, discounted_factor, ppo_steps, ppo_clip, trace_decay):
    policy.train()

    states = []
    rewards = []
    values = []
    log_prob_actions = []
    actions = []
    episode_reward = 0

    done, truncated = False, False 

    state, _ = env.reset() 

    while not done and not truncated: 
        state = torch.tensor(state).unsqueeze(0)
        states.append(state)

        action_pred, value_pred = policy(state)
        action_prob = F.softmax(action_pred, dim = -1)
        dist = distributions.Categorical(action_prob)
        action = dist.sample()
        log_prob_action = dist.log_prob(action)

        state, reward, done, truncated, _ = env.step(action.item())

        rewards.append(reward)
        actions.append(action)
        log_prob_actions.append(log_prob_action)
        values.append(value_pred)
        episode_reward += reward
    
    #chuyen sang tensor
    log_prob_actions = torch.cat(log_prob_actions)
    values = torch.cat(values).squeeze(-1)
    states = torch.cat(states)
    actions = torch.cat(actions)
    
    returns, advantages = compute_gae(values, rewards, discounted_factor, trace_decay)
    policy_loss, value_loss = update_policy(policy, optimizer, states, actions, returns, advantages, log_prob_actions, ppo_steps, ppo_clip)
    return policy_loss, value_loss, episode_reward

In [9]:
def compute_gae(values, rewards, discounted_factor, trace_decay):
    advantages = []
    gae = 0
    next_value = 0
    for reward, value in zip(reversed(rewards), reversed(values)):
        # Tinhs 1-step TD error
        delta = reward + discounted_factor * next_value - value.item()
        
        # Tich luy TD error voi trace_decay 
        gae = delta + discounted_factor * trace_decay * gae 
        next_value = value.item()
        advantages.insert(0, gae)
    advantages = torch.tensor(advantages, dtype = torch.float32) # N

    returns = advantages + values.detach() # N + N broadcast

    #normalize cho on dinh khi train 
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

    return advantages, returns

In [10]:
def update_policy(policy, optimizer, states, actions, returns, advantages, log_prob_actions, ppo_steps, ppo_clip):
    total_policy_loss = 0
    total_value_loss = 0
    
    #detacch 
    states = states.detach()
    advantages = advantages.detach()
    returns = returns.detach()
    actions = actions.detach()
    log_prob_actions = log_prob_actions.detach()

    for _ in range(ppo_steps):

        # get new log prob action for all input states
        action_pred, value_pred = policy(states)
        value_pred = value_pred.squeeze(-1)

        action_prob = F.softmax(action_pred, dim = -1)
        dist = distributions.Categorical(action_prob)
        entropy = dist.entropy().mean()
        next_log_prob_actions = dist.log_prob(actions)

        # tinh ratio between policy and old_policy 
        policy_ratio = (next_log_prob_actions - log_prob_actions).exp()
        policy_loss1 = policy_ratio * advantages 
        policy_loss2 = advantages * torch.clamp(policy_ratio,min = 1.0 - ppo_clip,max = 1.0 + ppo_clip)

        policy_loss = - torch.min(policy_loss1, policy_loss2).mean()
        policy_loss = policy_loss - 0.01 * entropy

        value_loss = F.smooth_l1_loss(returns, value_pred).mean()

        optimizer.zero_grad()
        policy_loss.backward()
        value_loss.backward()

        optimizer.step()
        total_policy_loss += policy_loss.item() 
        total_value_loss += value_loss.item() 

    return total_policy_loss, total_value_loss 


In [11]:
def evaluate(env, policy):
    policy.eval()
    done, truncated = False, False 
    episode_reward = 0 

    state, _ = env.reset()
    while not done and not truncated:
        with torch.no_grad():
            state = torch.tensor(state).unsqueeze(0)
            action_pred, _ = policy(state)
        action_prob = F.softmax(action_pred, dim = -1)
        action = torch.argmax(action_prob, dim = -1)
        state, reward, done, truncated, _ = env.step(action.item())
        episode_reward += reward 
    return episode_reward 

    

In [12]:
MAX_EPISODES = 500
DISCOUNT_FACTOR = 0.99
N_TRIALS = 25
REWARD_THRESHOLD = 500
PRINT_EVERY = 10
PPO_STEPS = 3
PPO_CLIP = 0.2
TRACE_DECAY = 0.99

train_rewards = []
test_rewards = []

for episode in range(1, MAX_EPISODES+1):
    
    policy_loss, value_loss, train_reward = train(train_env, policy, optimizer, DISCOUNT_FACTOR, PPO_STEPS, PPO_CLIP, TRACE_DECAY)
    
    test_reward = evaluate(test_env, policy)
    
    train_rewards.append(train_reward)
    test_rewards.append(test_reward)
    
    mean_train_rewards = np.mean(train_rewards[-N_TRIALS:])
    mean_test_rewards = np.mean(test_rewards[-N_TRIALS:])
    
    if episode % PRINT_EVERY == 0:
    
        print(f'| Episode: {episode:3} | Mean Train Rewards: {mean_train_rewards:5.1f} | Mean Test Rewards: {mean_test_rewards:5.1f} |')
    
    if mean_test_rewards >= REWARD_THRESHOLD:
        
        print(f'Reached reward threshold in {episode} episodes')
        
        break

| Episode:  10 | Mean Train Rewards:  24.7 | Mean Test Rewards:  11.4 |
| Episode:  20 | Mean Train Rewards:  27.5 | Mean Test Rewards:  60.1 |
| Episode:  30 | Mean Train Rewards:  31.4 | Mean Test Rewards: 115.8 |
| Episode:  40 | Mean Train Rewards:  33.0 | Mean Test Rewards: 143.8 |
| Episode:  50 | Mean Train Rewards:  37.3 | Mean Test Rewards: 121.0 |
| Episode:  60 | Mean Train Rewards:  38.9 | Mean Test Rewards: 131.3 |
| Episode:  70 | Mean Train Rewards:  40.1 | Mean Test Rewards: 170.3 |
| Episode:  80 | Mean Train Rewards:  44.3 | Mean Test Rewards: 218.4 |
| Episode:  90 | Mean Train Rewards:  51.1 | Mean Test Rewards: 245.0 |
| Episode: 100 | Mean Train Rewards:  57.6 | Mean Test Rewards: 342.1 |
| Episode: 110 | Mean Train Rewards:  63.9 | Mean Test Rewards: 262.8 |
| Episode: 120 | Mean Train Rewards:  62.9 | Mean Test Rewards: 185.4 |
| Episode: 130 | Mean Train Rewards:  44.6 | Mean Test Rewards:  74.5 |
| Episode: 140 | Mean Train Rewards:  30.6 | Mean Test Rewards: 

# So sánh với model stable_baseline3

In [15]:
test_env = gym.make("CartPole-v1")
evaluate(test_env, policy)

500.0

In [13]:
from stable_baselines3 import PPO 

env = gym.make("CartPole-v1")

model = PPO("MlpPolicy", env, verbose = 1)

model.learn(total_timesteps = 20000)

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.9     |
|    ep_rew_mean     | 21.9     |
| time/              |          |
|    fps             | 10756    |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 29.2        |
|    ep_rew_mean          | 29.2        |
| time/                   |             |
|    fps                  | 7206        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.008385381 |
|    clip_fraction        | 0.108       |
|    clip_range           | 0.2         |
|    entropy_loss   

In [14]:
for i in range(1,100 + 1):
    done, truncated = False, False 
    episode_reward = 0 
    obs, _ = env.reset()
    while not done and not truncated:
        action, _states = model.predict(obs, deterministic= True)
        obs, reward, done, truncated, _ = env.step(action)
        episode_reward += reward 

    print(f"|Episode {i} | episode_reward {episode_reward}")



|Episode 1 | episode_reward 500.0
|Episode 2 | episode_reward 500.0
|Episode 3 | episode_reward 500.0
|Episode 4 | episode_reward 500.0
|Episode 5 | episode_reward 500.0
|Episode 6 | episode_reward 500.0
|Episode 7 | episode_reward 500.0
|Episode 8 | episode_reward 500.0
|Episode 9 | episode_reward 500.0
|Episode 10 | episode_reward 500.0
|Episode 11 | episode_reward 500.0
|Episode 12 | episode_reward 500.0
|Episode 13 | episode_reward 500.0
|Episode 14 | episode_reward 500.0
|Episode 15 | episode_reward 500.0
|Episode 16 | episode_reward 500.0
|Episode 17 | episode_reward 500.0
|Episode 18 | episode_reward 500.0
|Episode 19 | episode_reward 500.0
|Episode 20 | episode_reward 500.0
|Episode 21 | episode_reward 500.0
|Episode 22 | episode_reward 500.0
|Episode 23 | episode_reward 500.0
|Episode 24 | episode_reward 500.0
|Episode 25 | episode_reward 500.0
|Episode 26 | episode_reward 500.0
|Episode 27 | episode_reward 500.0
|Episode 28 | episode_reward 500.0
|Episode 29 | episode_reward 